# Busca Semântica em Livros

Esse projeto apresenta o desenvolvimento de um modelo pré-RAG, com foco exclusivamente na recuperação de textos. o objetivo desse código é coletar textos de 15 livros e segmenta-los e transforma-los em vetores para permitir uma busca eficiente.
para ajudar na compreensão desse código, abaixo listo a ordem de cada passo do código:<br><br>
**1 - coleta e limpeza dos dados<br>**
**2 - análise dos textos coletados<br>**
**3 - Chunkerização<br>**
**4 - Embedding<br>**
**5 - salvamento dos dados<br>**
**6 - função de consulta<br>**
**7 - testes finais<br>**
**8 - melhorias**<br>
**9 - links externos**

importação das bibliotecas

In [1]:
import os
import re
import numpy as np
from bs4 import BeautifulSoup
import pickle

from llama_index.core.node_parser import SentenceSplitter
from llama_index.core import Document
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

livros = "book"  # pasta dos livros

ValueError: Your currently installed version of Keras is Keras 3, but this is not yet supported in Transformers. Please install the backwards-compatible tf-keras package with `pip install tf-keras`.

---

# 1 - coleta e limpeza dos dados

- analisando a estrutura dos livros, percebi que em alguns há um padrão de onde estão disponiveis os textos. boa parte dos textos dos livros estão escritos em uma `<div class="chapter">` e depois separado com `<p>` sem a presença de nenhuma classe. então para uma coleta mais precisa desses textos, desenvolvi uma condição que colete apenas conteudos de paragrafos presentes nessa div.
<br>
- há alguns livros que usam o padrão `<p class="drama">`. então criei uma condição que colete tanto com paragrafos sem classe quanto com paragrafos com classes **drama** .
- alguns livros usam `div id="chapter-x"` onde x vai aumentando conforme o capitulo. para resolver a coleta desses livros, criei uma condição que vai coletando a classe conforme o loop vai passando, utilizando uma variavel numerica para substituir o valor de x.
- alguns livros não apresentam um padrão nos textos, e para resolver isso, criei uma condição final que coleta todos os `<p>` dos textos, infelizmente, esses livros vão estar cheios de problemas, com varios textos desnecessarios, que infelizmente poderão gerar problemas na recuperação, mas talvez isso possa ser evitado com uma boa chunkerização.

In [ ]:
books = []                                                                   # lista de armazenamento dos livros

for file_name in os.listdir(livros):                                         # loop para realizar coleta de cada livro

    if file_name.endswith(".html"):                                          # condição para coletar apenas arquivos html

        html_file = os.path.join(livros, file_name)                          # variavel contendo o livro e o local de armazenamento

        with open(html_file, "r", encoding="utf-8", errors="ignore") as f:   # abre o arquivo em modo leitura 
            html_content = f.read()                                          # realiza a leitura dos arquivos

        soup = BeautifulSoup(html_content, "lxml")                           # crie obj soup para mapear elementos html
        textos = []                                                          # lista para armazenar os textos coletados de cada livro

        # condição para coleta de textos dentro da div chapter
        chapters = soup.find_all("div", class_="chapter")                    # variavel que coleta as partes de codigos com as determinadas divs
        if chapters:                                                      
            for chapter in chapters:
                for p in chapter.find_all("p"):                              # coleta todas as tags <p> dentro dos livros

                    if not p.attrs or p.get("class") == ["drama"]:           # filtra as tags <p> que não contenham classes ou que contem a classe drama
                        text = p.get_text(" ", strip=True)                   # extrai todos os textos presentes nessas tags, e remove espaços extras
                        if text:                                             # se o texto coletado não for vazio, o texto coletado e armazenado na lista textos
                            textos.append(text)

        # condição caso os textos não estejam salvos dentro de uma div com classe chapter
        if not textos:                                                       # condição para realizar outra coleta caso a variavel textos esteja vazia 
            chapter_num = 1                                                  # variavel inteira para coletar os capitulos 
            while True:                                                      # loop para coletar todo o conteudo do livro
                chapter_div = soup.find("div", id=f"chapter-{chapter_num}")  # variavel para coletar as partes do livro com uma vid de ID chapter-x
                if not chapter_div:                                          # caso o livro não tenha nenhuma tag, ele finaliza o loop
                    break                       
                
                for p in chapter_div.find_all("p"):                          # loop para coletar todos os <p> das divs coletadas

                    if not p.attrs:                                          # variavel para coletar apenas os <p> sem nenhum atributo
                        text = p.get_text(" ", strip=True)
                        if text:                                             # condição para salvar todos os textos coletados
                            textos.append(text)                                 

                chapter_num += 1                                             # variavel numerica para coletar cada capitulo dos livros

        # ultima condição, ela simplesmente coleta todos os textos que estão dentro de <p>. essa condição serve apenas para coletar os
        # livros que não foram coletados com as condições anteriores
        if not textos:                                                       # condição para caso os certos livros ainda esteejam vazios
            for p in soup.find_all("p"):
                if not p.attrs:                                              # loop para coletar todos os <p>
                    text = p.get_text(" ", strip=True)                       # simplesmente coleta todos os textos dentro de <p>
                    if text:    
                        textos.append(text)

        # aqui realizo uma limpeza em cada texto coletado, removendo simbolos desnecessarios
        if textos:                                                           # caso o texto coletado não esteja vazio             
            texto_limpo = " ".join(textos)                                   # variavel vazia que receberá os textos
 
            texto_limpo = re.sub(r'[^a-zA-Z0-9áéíóúâêîôûãõçÁÉÍÓÚÂÊÎÔÛÃÕÇ\s.,;:!?\-\'\"«»]', '', texto_limpo)  # remoção dos caracteres indesejados

            texto_limpo = re.sub(r'\s+', ' ', texto_limpo)                   # remove espaços extras vazios
            texto_limpo = texto_limpo.strip()                                # para remover todos os espaços duplos
        else:                                                                # condição para caso nenum texto tenha sido coletado, a lista ficará vazia
            texto_limpo = ""                        

        books.append({                                                       # dicionario com o titulo dos livros + os textos coletados
            "title": os.path.splitext(file_name)[0],                         # coleta e salva o titulo do livro presente no nome do arquivo
            "text": texto_limpo                                              # salva os textos limpos coletados
        })

---

# 2 - Analise dos textos coletados

### 2.1 - tamanho dos textos em relação os livros

para verificar se a extração não foi problematica, eu realizo essa analise sobre a porcentagens de caracteres que foram coletados em relação ao livro original. 
- uma baixa porcentagem pode indicar que muito conteudo foi removido, o que pode indicar que partes relevantes do livros podem ter sido ignorados.
- uma porcentagem de 70 a 90 pode indicar que a coleta foi eficiente.
- uma porcentagem acima de 90 pode indicar que muitas coisas desnecessarias foram coletadas.

como alguns livros não seguiram o padrão, a porcentagem de coleta pode ter saido extremamente alta. e importante notar que alguns livros apresentam varios capitulos irrelevantes para esse projeto, e isso pode diminuir drasticamente a porcentagem.

In [ ]:
for book in books:                                                          # loop para coletar cada item do dicionario

    html_file = os.path.join(livros, book["title"] + ".html")               # coleta o local de cada livro

    with open(html_file, "r", encoding="utf-8", errors="ignore") as f:      # leitura dos livros
        html_content = f.read()

    tamanho_html = len(html_content)                                        # variavel que contabiliza o numero de caracteres presente nos livros completos
    tamanho_texto = len(book["text"])                                       # variavel da somatoria de todos os caracteres dos textos coletados

    percentual = (tamanho_texto / tamanho_html) * 100                       # porcentagem emtre os caracrteres 

    print(f"Livro: {book['title']}")
    print(f"Tamanho original: {tamanho_html:,} caracteres")
    print(f"Texto extraído: {tamanho_texto:,} caracteres")
    print(f"porcentagem de caracteres coletados: {percentual:.2f}%")
    print("-" * 60)

### 2.2 - analise de caracteres indesejaveis

há muitos simbolos indesejaveis nos livros, tabelas, imagens, tags HTML. no codigo de coleta, ja realizei limpezas desses caracteres indesejaveis, e devido a forma como coleto apenas conteudos dentro de `<p>`, não haverá tags desnecessarias. porem, para ter a certeza que não tem nenhuma, criei a celula abaixo para realizar uma analise de tags HTML e simbolos indejesados em cada livro.

In [ ]:
textos_vistos = set()                                  # cria um conjunto para armazenar os textos

for book in books:

    titulo = book["title"]                             # variavel com o titulo dos livros
    texto = book["text"]                               # variavel com o texto dos livros

    if not texto.strip():                              # verifica se o texto esta vazio
        print(f"[VAZIO] {titulo}")
        continue

    if texto in textos_vistos:                         # verifica se o texto coletado ja foi relido anteriormente
        print(f"[DUPLICADO] {titulo}")

    textos_vistos.add(texto)

### 2.3 demonstrativo dos livros

In [ ]:
for book in books:
    print(f"Livro: {book['title']}")
    print(f"texto: {book['text']}")
    print("-"*60)

---

# 3 - Chunckerização

durante a o desenvolvimento dessa etapa, encontrei um artigo comentando sobre 7 estrategias de chunk, das que achei mais interessante para esse desafio foram:<br>
**1 - recursive character splitting<br>**
**2 - sentence-based chunking<br>**
**3 - semantic chunking<br>**
para realizaçaõ dessa etapa, eu escolhi o modelo **Sentence-Based Chunking**. eu escolhi esse modelo devido a sua caraceristica principal, por ser um modelo que foca justamente na sentença, esse modelo parece garantir alta conpatibilidade na separaçaõ de frases, garantindo chunks com significados mais preservados. o modelo **recursive character splitting** parece ser muito bom, mas devido a sua forma de separar as frases, ele pode acabar separando um paragrafo bem estruturado em dois chunks, e isso pode gerar problema na comprensão das frases. alias, o modelo escolhido, pode gerar chunks com tamnahos extremamente direfentes para garantir que cada um tenha um significado mais adequado. o terceiro modelo analisado, eu preferi deixar de lado devido o uso de LLM, acredito que teria um custo muito alto para um projeto pequeno.

In [ ]:
splitter = SentenceSplitter(                                 # variavel para o Sentece-Based Chunking
    chunk_size=128,                                          # configuração de chunks
    chunk_overlap=30                                         # configuração de overlaps
)

chunks_dataset = []                                          # lista de armazenamento dos chunks           

for livro_idx, book in enumerate(books):                      

    titulo_livro = book["title"]                            # variavel para coletar o titulo dos livros                       
    texto_completo = book["text"]                           # variavel para coletar o texto dos livros

    if not texto_completo.strip():                          # apenas para pular livros que estejam vazios
        continue

    llama_doc = Document(text=texto_completo)               # converte o texto para documento para ser utilizado no modelo
                                                            # o Sentence Splitter só parece funcionar com documentos
    nodes = splitter.get_nodes_from_documents([llama_doc])  # realiza a divisão dos documentos em chunks

    for chunk_idx, node in enumerate(nodes):                # loop para salvar os chunks no dicionario

        chunk_data = {                                            # esse dicionario que armazenará cada chunk e suas informações                                            
            "chunk_id": f"livro{livro_idx}_chunk_{chunk_idx}",    # ID do chunk        
            "titulo": titulo_livro,                               # titulo do livro        
            "texto": node.text                                    # texto do chunk
        }

        chunks_dataset.append(chunk_data)                      
        
print(f"Total de livros processados: {len(books)}")
print(f"Total de chunks armazenados: {len(chunks_dataset)}")

#### 3.1 - analise do chunk

demonstração de quantos chunks cada livro recebeu

In [ ]:
for livro_idx, book in enumerate(books):

    total_chunks = len([
        chunk for chunk in chunks_dataset 
        if chunk["chunk_id"].startswith(f"livro{livro_idx}_")
    ])
    
    print(f"Livro {livro_idx}: {book['title']}")
    print(f"Quantidade de chunks: {total_chunks}")
    print("-" * 80)

aqui podemos analisar cada livro e seus chunks

In [ ]:
livro_escolhido = 3                                                     # variavel de cada livro
chunk_escolhido = 10                                                    # variavel para escolha dos chunks

chunk_id_procurado = f"livro{livro_escolhido}_chunk_{chunk_escolhido}"

for chunk in chunks_dataset:

    if chunk["chunk_id"] == chunk_id_procurado:

        print(f"Título: {chunk['titulo']}")
        print(f"Chunk ID: {chunk['chunk_id']}")
        print(chunk["texto"])
        break

# 4 - embeddings

* durante o processo de embeddings, eu estava utilizando o modelo BGE-Large. infelizmente o treino dele durava muito e como eu precisava realizar mudanças nos chunks, overlaps e tambem nas tecnincas de chunks, optei por usar o all-mini. esse modelo e mais basico, porem mais rapido em relação ao BGE. <br>
* durante o processo de embeddings, eu iniciei utilizando 512 chunks com 20% de overlap. porem, esses valores estavam gerando textos grandes e problematicos, para ter uma resposta mais precisa, optei por usar 256, com esses valores, ainda gerava textos grandes e tambem sei resultados adequados. no final, modifiquei os chunks para 128 com overlap de 30. e assim obtive respostas mais "precisas".<br>
* com a configuração de 128 chunks, consegui resultados mais confiaveis nas perguntas faceis, porem algumas perguntas dificeis não estavam coerentes, e por conta disso, optei por mudar o modelo de embedding para o all-mpnet, que conforme uma pesquisa que fiz, parece ter um treinamento um pouco mais rapido em relação ao BGE large e um resultado mais preciso em relação ao all-mini.

In [ ]:
modelo = SentenceTransformer("all-mpnet-base-v2")                     # carrega o modelo all-mini

textos_chunks = [chunk['texto'] for chunk in chunks_dataset]         # lista para extrair os textos chunlerizados

embedding = modelo.encode(                                           # transforma a linha de textos em embeddings
    textos_chunks,                                             
    show_progress_bar=True,                                         
     # apenas para mostrar uma barra de treino
    convert_to_numpy=True,                                           # converte os valores em uma matriz de numpy
    normalize_embeddings=True                                        # normalização dos embeddings
)

for idx, chunk in enumerate(chunks_dataset):
    chunk["embedding"] = embedding[idx]                              # adição dos embeddings no dicionario

# 5 - salvamento dos dados

In [ ]:
dados_finais = "treino.pkl"          

with open(dados_finais, "wb") as f:
    pickle.dump(chunks_dataset, f)

# 6 - Função de consulta

criei essa função para coletar os valores de top 1 e top 10. para facilitar na amostragem dos chunks coletados

In [ ]:
with open("treino.pkl", "rb") as f:                      # para reutilizar o arquivo treino.pkl
    chunks_dataset = pickle.load(f)
modelo = SentenceTransformer("all-mpnet-base-v2")        # para carregar o modelo all-mpnet

In [ ]:
def resultado(pergunta_usuario, chunks_dataset, modelo):                            # função para receber a pergunta, e enviar o top 1 e os top 10 embeddings

    embedding_pergunta = modelo.encode(
        pergunta_usuario,                                                           # entrada da pergunta
        convert_to_numpy=True,                                                      # apenas conversão para lidar com numpy
        normalize_embeddings=True                                                   # normalização dos embeddings
    ).reshape(1, -1)                                                                # ajusta a entrada para uma matriz

    embeddings_treino = np.array( [chunk["embedding"] for chunk in chunks_dataset]) # coleta os embeddings do dataset

    scores_similaridade = cosine_similarity(                                        # realiza o calculo de similaridade do cosseno entre a pergunta e os chunks
                    embedding_pergunta,
                    embeddings_treino
                    )[0]              

    chunks_com_score = []                                                           # lista com chunks e os scores

    for idx, chunk in enumerate(chunks_dataset):                                            

        chunk_copia = chunk.copy()                                                  # apenas para criar uma copia e não correr o risco de substituir o orginal
 
        chunk_copia["score_similaridade"] = (scores_similaridade[idx])              # salva o score de similaridade na nova lista

        chunks_com_score.append(chunk_copia)

    chunks_ordenados = sorted(                                                     # ordena os chunks do maior para o menor em relação ao score de similaridade
        chunks_com_score,
        key=lambda x: x["score_similaridade"],
        reverse=True
    )

    chunk_top_1 = chunks_ordenados[0]                                              # variavel para pegar o chunk mais relevante

    top_10_chunks = chunks_ordenados[:10]                                          # variavel para pegar uma lista dos 10 chunks mais relevantes

    return chunk_top_1, top_10_chunks

# 7 - testes finais

In [ ]:
pergunta = "In The Importance of Being Earnest, what are the names used by Jack when he creates his false identity?"                                                 # escreva a pergunta aqui
chunk_final, top_10_chunks = resultado(pergunta, chunks_dataset, modelo)               # variaveis para receber o melhor chunk e os 10 mais relevantes

In [ ]:
print(f"Chunk mais relevante - Score: {chunk_final['score_similaridade']:.2f}")
print(f"Livro: {chunk_final['titulo']} | ID do Chunk: {chunk_final['chunk_id']}")
print(chunk_final['texto'])

Chunk mais relevante - Score: 0.63
Livro: Frankenstein | ID do Chunk: livro3_chunk_151
So much has been done, exclaimed the soul of Frankensteinmore, far more, will I achieve; treading in the steps already marked, I will pioneer a new way, explore unknown powers, and unfold to the world the deepest mysteries of creation. I closed not my eyes that night. My internal being was in a state of insurrection and turmoil; I felt that order would thence arise, but I had no power to produce it. By degrees, after the mornings dawn, sleep came. I awoke, and my yesternights thoughts were as a dream.


In [6]:
print("os 10 melhores chunks")
for posicao, chunk in enumerate(top_10_chunks, start=1):
    print(f"{posicao}º Lugar | Score: {chunk['score_similaridade']:.2f} | Livro: {chunk['titulo']} (ID: {chunk['chunk_id']})")
    print(f"Trecho: {chunk['texto']}")
    print("--"*60)

os 10 melhores chunks
1º Lugar | Score: 0.63 | Livro: Frankenstein (ID: livro3_chunk_151)
Trecho: So much has been done, exclaimed the soul of Frankensteinmore, far more, will I achieve; treading in the steps already marked, I will pioneer a new way, explore unknown powers, and unfold to the world the deepest mysteries of creation. I closed not my eyes that night. My internal being was in a state of insurrection and turmoil; I felt that order would thence arise, but I had no power to produce it. By degrees, after the mornings dawn, sleep came. I awoke, and my yesternights thoughts were as a dream.
------------------------------------------------------------------------------------------------------------------------
2º Lugar | Score: 0.63 | Livro: Frankenstein (ID: livro3_chunk_939)
Trecho: I pitied Frankenstein; my pity amounted to horror; I abhorred myself. But when I discovered that he, the author at once of my existence and of its unspeakable torments, dared to hope for happiness, 

sobre os resultados finais, o chunk mais relevante nem sempre é o mais coerente com as perguntas, porem, ao analisar os top 10, percebi que alguns deles são mais coerentes com a pergunta.<br>
no geral, os resultados foram bons, tanto com as perguntas faceis quanto com as dificeis. 

---

# 8 - melhorias

sobre as melhorias, listei coisas que eu faria para tornar o projeto mais interessante:
* 1 - mudança na forma de realizar os scraps dos livros. infelizmente não consegui uma maneira mais eficiente de recuperar todos os conteudos relevantes, e por isso, alguns * livros ficaram cheios de textos inuteis, enquanto outros acabaram perdendo alguns conteudos relevantes.*
* 2 - não acredito que seja ruim para esse projeto, mas talvez a mudança da chunkerização melhorasse os resultados, isso por que o modelo sentence-based chunk tende a ter problemas com textos mal formatados.
* 3 - analisar melhor os valores de chunking e overlap, pois as mudanças que fiz, foi realizada de forma mais adequada no modelo all-mini. e talvez valores maiores seriam melhores para o all-mpnet.

# 9 - Links Externos

* https://www.firecrawl.dev/blog/best-chunking-strategies-rag --> usei para escolher as melhores estrategias de chunking
* https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2 --> usei para estudar um pouco sobre o all-mini
* https://huggingface.co/sentence-transformers/all-mpnet-base-v2 --> usei para o desenvolvimento do mpnet
